In [ ]:
# Install required packages (run this cell first if packages are not installed)
# !pip install selenium webdriver-manager beautifulsoup4 pandas requests

# Basketball Statistics Extractor

This notebook extracts basketball player statistics from the Basketball Stats Vlaanderen website.

In [1]:
# Import Required Libraries
import requests
from bs4 import BeautifulSoup
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time

In [2]:
def extract_basketball_stats_table(url):
    """
    Extract the basketball statistics table from the given URL using Selenium
    
    Args:
        url: The basketball stats URL
        
    Returns:
        pandas.DataFrame: The extracted table data
    """
    
    # Setup Chrome options
    chrome_options = Options()
    chrome_options.add_argument("--headless")  # Run in background
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--window-size=1920,1080")
    chrome_options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    
    # Create driver
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)
    
    try:
        print(f"Loading URL: {url}")
        driver.get(url)
        
        # Wait for page to load and table to appear
        print("Waiting for page to load...")
        time.sleep(5)
        
        # Try to wait for table to be present
        try:
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.TAG_NAME, "table"))
            )
        except:
            print("Table not found with WebDriverWait, continuing...")
        
        # Get page source after JavaScript execution
        html_content = driver.page_source
        
        # Parse with BeautifulSoup
        soup = BeautifulSoup(html_content, 'html.parser')
        
        # Find all tables and look for the one with basketball stats
        tables = soup.find_all('table')
        print(f"Found {len(tables)} tables on the page")
        
        # Look for table with 'Competitie' header
        target_table = None
        for i, table in enumerate(tables):
            headers = [th.get_text(strip=True) for th in table.find_all('th')]
            print(f"Table {i+1} headers: {headers}")
            
            if headers and any('Competitie' in header for header in headers):
                target_table = table
                print(f"Found target table at index {i+1}")
                break
        
        if target_table:
            # Try to extract with pandas
            df = pd.read_html(str(target_table))[0]
            print(f"Successfully extracted table with shape: {df.shape}")
            return df
        else:
            print("Table with 'Competitie' header not found")
            # Let's also try looking for tables with specific content
            for i, table in enumerate(tables):
                table_text = table.get_text()
                if 'Alle wedstrijden' in table_text or 'U18 Niveau' in table_text:
                    print(f"Found table with basketball content at index {i+1}")
                    df = pd.read_html(str(table))[0]
                    return df
            return None
            
    except Exception as e:
        print(f"Error occurred: {e}")
        return None
    
    finally:
        driver.quit()

In [3]:
# Diagnostic: Check what might be blocked
import subprocess
import sys

def check_blocking_issues():
    print("Checking for potential blocking issues...")
    
    # Check if we can reach the website with simple requests
    try:
        import requests
        url = "https://app.basketballstatsvlaanderen.be/players/BVBL744354?season=2425"
        response = requests.get(url, timeout=10)
        print(f"✅ Basic HTTP request successful (status: {response.status_code})")
        print(f"   Content length: {len(response.text)} characters")
    except Exception as e:
        print(f"❌ Basic HTTP request failed: {e}")
        print("   This suggests network/firewall blocking")
    
    # Check if Chrome is available
    try:
        from selenium import webdriver
        from selenium.webdriver.chrome.options import Options
        options = Options()
        options.add_argument("--headless")
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
        
        print("✅ Selenium imports successful")
        
        # Try to create a driver (this will download ChromeDriver if needed)
        from webdriver_manager.chrome import ChromeDriverManager
        from selenium.webdriver.chrome.service import Service
        
        service = Service(ChromeDriverManager().install())
        print("✅ ChromeDriver manager successful")
        
    except Exception as e:
        print(f"❌ Selenium/Chrome setup failed: {e}")
        print("   Possible issues:")
        print("   - Chrome not installed")
        print("   - Antivirus blocking ChromeDriver")
        print("   - Corporate firewall blocking downloads")

check_blocking_issues()

Checking for potential blocking issues...
✅ Basic HTTP request successful (status: 200)
   Content length: 1283867 characters
✅ Selenium imports successful
✅ ChromeDriver manager successful


In [4]:
# Set the URL for the basketball player statistics
url = "https://app.basketballstatsvlaanderen.be/players/BVBL744354?season=2425"

print("Starting extraction process...")
print("Note: This may take a few moments as we need to load the page with Selenium")

# Extract the table
df = extract_basketball_stats_table(url)

if df is not None:
    print("\n" + "="*60)
    print("Successfully extracted basketball statistics table:")
    print("="*60)
    display(df)
    print("="*60)
    print(f"Table shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
else:
    print("Failed to extract table")

Starting extraction process...
Note: This may take a few moments as we need to load the page with Selenium
Loading URL: https://app.basketballstatsvlaanderen.be/players/BVBL744354?season=2425
Waiting for page to load...
Found 4 tables on the page
Table 1 headers: []
Table 2 headers: ['Competitie', 'Punten', 'F']
Found target table at index 2
Successfully extracted table with shape: (1, 3)


C:\Users\StijnHuysman\AppData\Local\Temp\ipykernel_33320\4146897286.py:63: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(target_table))[0]



Successfully extracted basketball statistics table:


,Competitie,Punten,F
0,Geen resultaten gevonden,Geen resultaten gevonden,Geen resultaten gevonden


Table shape: (1, 3)
Columns: ['Competitie', 'Punten', 'F']


In [10]:
# Save the extracted data to CSV file
if df is not None:
    output_file = "basketball_stats.csv"
    df.to_csv(output_file, index=False)
    print(f"Table saved to: {output_file}")
    
    # Display basic statistics
    print("\nTable Info:")
    print(df.info())
    
    print("\nFirst few rows:")
    display(df.head())

Table saved to: basketball_stats.csv

Table Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Competitie  0 non-null      object
 1   Punten      0 non-null      object
 2   F           0 non-null      object
dtypes: object(3)
memory usage: 132.0+ bytes
None

First few rows:


,Competitie,Punten,F


In [11]:
# Optional: Beautify the dataframe display
if df is not None:
    # Style the dataframe for better visualization
    styled_df = df.style.set_properties(**{
        'text-align': 'center',
        'font-size': '12px'
    }).set_table_styles([
        {'selector': 'th', 'props': [('text-align', 'center'), ('font-weight', 'bold')]}
    ])
    
    display(styled_df)

AttributeError: The '.style' accessor requires jinja2